In [1]:
import os
import matplotlib.pyplot as plt

import rasterio 
import pandas as pd
import geopandas as gpd
from scipy.ndimage import gaussian_filter
from shapely.geometry import Polygon
from pathlib import Path
from tqdm.auto import tqdm
from rasterio.windows import from_bounds
from rasterio.enums import Resampling

from torch import Tensor

In [2]:
import sys
sys.path.append('../')
from utils import get_tiles_as_gdf, extract_sample, get_window

In [3]:
path = "/home/jan/Documents/EPFL/project_2/data/labels/dataset_v4/labels_55k.csv"

labels = pd.read_csv(path)


In [4]:
def _get_image_bbox(image_path):


    with rasterio.open(image_path) as src:
        lower_left = src.xy(109, 10, offset='ll')
        upper_left = src.xy(10, 10, offset='ul')

        lower_right = src.xy(109, 109, offset='lr')
        upper_right = src.xy(10, 109, offset='ur')
    poly = Polygon([lower_left, upper_left, upper_right, lower_right])
    
    
    return poly
    
    #print(src.xy(10, 10, offset='ul'))
    
    
    
def resample(src, width, height):

    # resample data to target shape
    data = src.read(
        out_shape=(
            src.count,
            height,
            width
        ),

        resampling=Resampling.nearest
    )

    # scale image transform
    out_transform = src.transform * src.transform.scale(
        (src.width / data.shape[-1]),
        (src.height / data.shape[-2])
    )

    out_profile = src.profile
    out_profile.update(
        {
            'width': width,
            'height': height,
            'transform': out_transform
            }
        )

    return data, out_profile

In [5]:
forestloss_dir = '../../Vancutsem_yearloss'
forestloss_tiles = get_tiles_as_gdf(forestloss_dir)
image_dir = '/home/jan/Documents/EPFL/project_2/data/images/dataset_v4/55k/SR_four_images'
out_dir = '../data/forestloss_tiles/dataset_v4_test'
target_shape = (33,33)

Path(out_dir).mkdir(parents=True, exist_ok=True)

#shapes = {}
sampleids_to_skip = []

ids = []

for idx, row in tqdm(labels.iterrows()):
    
    sampleid = str(row.sampleid)
    sampleid_dir = os.path.join(image_dir,sampleid)
    img_filename = os.listdir(sampleid_dir)[0]
    image_path = os.path.join(sampleid_dir, img_filename)
    
    poly = _get_image_bbox(image_path)
    
    bounds = poly.bounds
    
    
    intersecting_tiles = forestloss_tiles.loc[forestloss_tiles.intersects(poly)].filename.values
    
    
    if len(intersecting_tiles) != 1: #Bordering multiple tiles, skipping
        print(sampleid)
        sampleids_to_skip.append(int(sampleid))
    
    if len(intersecting_tiles) == 0:
        print(f'does not intersect with any forest loss tile: sampleid {sampleid}')
        continue
    
    tmf_filename = intersecting_tiles[0]

    with rasterio.open(tmf_filename) as src:

        window = from_bounds(*bounds, src.transform)

        # Read the data from the defined window
        data = src.read(window=window)
        
        #if data.shape == (1,33,37):
        #    ids.append(sampleid)
        #if len(intersecting_tiles) != 1:
        #    if data.shape in shapes.keys():
        #        shapes[data.shape] += 1
        #    else:
        #        shapes[data.shape] = 1
        #print(f'intersecting tiles: {len(intersecting_tiles)}', f' patch shape: {data.shape}')
        
            
            
        #min_val = 2000
        #max_val = 2020
        #data[(data < min_val) | (data > max_val)] = 0
        #data[(data >= min_val) & (data <= max_val)] = 1
        #data = data.astype('float')
        #data = gaussian_filter(data, sigma=1)
        #data = Tensor(data)

        # Get the profile (metadata) from the source raster and update it for the output
        profile = src.profile
        profile.update({
            'height': window.height,
            'width': window.width,
            'dtype' : "float32",
            'transform': rasterio.windows.transform(window, src.transform)
        })
        
        
        out_path = os.path.join(out_dir, f'{sampleid}.tif')

        # Write the data to a new raster file
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data)
          
        
        #Do it the same way as the Sentinel-2 images processing
        if (data.shape[1], data.shape[2]) != target_shape:
            with rasterio.open(out_path) as src:
                out_image, out_profile = resample(src, target_shape[0], target_shape[1])
            with rasterio.open(out_path, "w", **out_profile) as sink:
                sink.write(out_image)
            
    

0it [00:00, ?it/s]

In [12]:
labels_filtered = labels.loc[~labels.sampleid.isin(sampleids_to_skip)].reset_index(drop=True)
labels_filtered.to_csv(path, index=False)

In [20]:
xx = gpd.GeoDataFrame(labels_filtered)
xx['geometry'] = gpd.GeoSeries.from_wkt(xx['geometry'])
xx = xx.set_geometry("geometry")
xx.to_file('../data/tmp/labels_v4_55k.gpkg')